# TekaRx full-graph restore and 3D visualization

This notebook restores the exact verified `gnn-full` topology from Google Drive and creates a bounded, interactive 3D view. The complete graph has millions of nodes and tens of millions of edges, so attempting to draw every element would freeze the browser. The visualization is a seeded sample; its manifest reports the sample as a percentage of the complete graph.

The restore is intentionally topology-only: it copies patient IDs, targets, split IDs, drug features/metadata, and both edge-index arrays. It does **not** copy the 3.86 GB `patient_x` matrix because rendering does not use it. This graph is for research exploration and is not a diagnosis.

## 1. Start a Colab runtime

A CPU runtime is sufficient. GPU does not accelerate file restoration or browser rendering. Run the cells from top to bottom; rerunning them is safe because complete files are reused after size verification.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Google Drive mounted.")

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import socket
import subprocess
import sys
from datetime import UTC, datetime

REPO_URL = "https://github.com/matthew-sudo2/Teka-Rx.git"
REPO_DIR = Path("/content/Teka-Rx")
GIT_REF = "main"
DRIVE_DATA = Path("/content/drive/MyDrive/Teka-Rx-full/data")
LOCAL_DATA = Path("/content/tekarx-full-visualization/data")

SPLIT = "validation"  # train, validation, or test
PATIENTS = 500            # command safety limit: 1..500
TOP_DRUGS = 200           # command safety limit: 1..250
SEED = 42
DOWNLOAD_TO_BROWSER = False

if SPLIT not in {"train", "validation", "test"}:
    raise ValueError(f"Invalid SPLIT: {SPLIT}")
if not 1 <= PATIENTS <= 500 or not 1 <= TOP_DRUGS <= 250:
    raise ValueError("PATIENTS must be 1..500 and TOP_DRUGS must be 1..250")
if not DRIVE_DATA.is_dir():
    raise FileNotFoundError(f"Missing Drive data directory: {DRIVE_DATA}")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
print(f"Drive data: {DRIVE_DATA}")
print(f"Local workspace: {LOCAL_DATA}")
print(f"Local free space: {shutil.disk_usage('/content').free / 2**30:.2f} GiB")

In [ ]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def copy_with_progress(source: Path, destination: Path, *, expected_size: int | None = None) -> None:
    from tqdm.auto import tqdm

    if not source.is_file():
        raise FileNotFoundError(source)
    source_size = source.stat().st_size
    if expected_size is not None and source_size != expected_size:
        raise RuntimeError(f"Drive artifact size mismatch: {source}")
    if destination.is_file() and destination.stat().st_size == source_size:
        print(f"REUSE {destination.name} ({source_size / 2**20:.1f} MiB)")
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".copying")
    temporary.unlink(missing_ok=True)
    with source.open("rb") as reader, temporary.open("wb") as writer, tqdm(
        total=source_size, unit="B", unit_scale=True, desc=destination.name
    ) as progress:
        while chunk := reader.read(8 * 1024 * 1024):
            writer.write(chunk)
            progress.update(len(chunk))
        writer.flush()
        os.fsync(writer.fileno())
    if temporary.stat().st_size != source_size:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(f"Incomplete copy: {destination}")
    os.replace(temporary, destination)


def run_live(arguments: list[str], *, cwd: Path | None = None) -> None:
    print("$", " ".join(str(value) for value in arguments), flush=True)
    process = subprocess.Popen(arguments, cwd=cwd, text=True)
    return_code = process.wait()
    if return_code:
        raise RuntimeError(f"Command failed with exit status {return_code}")

print("Helpers ready.")

## 2. Install the visualization command

This uses a plain Git URL (not a Markdown-formatted URL) and installs the checked-out repository into the current Colab kernel.

In [ ]:
if not (REPO_DIR / ".git").is_dir():
    run_live(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    run_live(["git", "-C", str(REPO_DIR), "fetch", "--prune", "origin"])

if GIT_REF == "main":
    run_live(["git", "-C", str(REPO_DIR), "checkout", "--detach", "origin/main"])
else:
    run_live(["git", "-C", str(REPO_DIR), "checkout", "--detach", GIT_REF])

run_live([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])
help_result = subprocess.run(
    [sys.executable, "-m", "tekarx.cli", "visualize-graph", "--help"],
    text=True, capture_output=True, check=False,
)
if help_result.returncode or "--layout" not in help_result.stdout:
    raise RuntimeError(
        "This repository revision does not contain the full-graph visualizer. "
        "Push the latest TekaRx changes, restart the runtime, and rerun this notebook."
    )
print(help_result.stdout)

## 3. Verify and restore the exact full-graph topology

The durable `_GRAPH_SUCCESS.json` marker is authoritative. Incomplete `.uploading-*` or `.downloading-*` directories are ignored.

In [ ]:
marker_path = DRIVE_DATA / "processed" / "_GRAPH_SUCCESS.json"
if not marker_path.is_file():
    raise FileNotFoundError(
        f"Missing verified graph marker: {marker_path}. Run Stage 2 of train_full_colab.ipynb first."
    )
graph_marker = load_json(marker_path)
required_marker_values = {
    "stage": "graph_complete",
    "checkpoint": "verified_complete",
    "split_preset": "gnn-full",
    "graph_storage": "tekarx.memmap_graph",
}
for key, expected in required_marker_values.items():
    if graph_marker.get(key) != expected:
        raise RuntimeError(f"Graph marker {key}={graph_marker.get(key)!r}; expected {expected!r}")

GRAPH_CHECKPOINT_ID = graph_marker["graph_checkpoint_id"]
checkpoint_relative = Path(graph_marker["checkpoint_dir"])
if checkpoint_relative.is_absolute() or not checkpoint_relative.parts or checkpoint_relative.parts[0] != "graph_checkpoints" or ".." in checkpoint_relative.parts:
    raise RuntimeError(f"Unsafe graph checkpoint path: {checkpoint_relative}")
drive_graph_dir = DRIVE_DATA / "processed" / checkpoint_relative
drive_array_dir = drive_graph_dir / "tekarx_graph_arrays"
drive_array_manifest_path = drive_array_dir / "manifest.json"
if not drive_array_manifest_path.is_file():
    raise FileNotFoundError(drive_array_manifest_path)
array_manifest = load_json(drive_array_manifest_path)
if array_manifest.get("format") != "tekarx.memmap_graph":
    raise RuntimeError("Checkpoint is not a TekaRx memory-mapped graph.")
counts = array_manifest["counts"]
print(json.dumps({"checkpoint_id": GRAPH_CHECKPOINT_ID, **counts}, indent=2))

In [ ]:
required_arrays = (
    "patient_primaryid", "patient_y", "patient_split_id",
    "edge_patient_index", "edge_drug_index", "drug_x",
)
local_graph_dir = (
    LOCAL_DATA / "processed" / "graph_visualization_checkpoints" / GRAPH_CHECKPOINT_ID
)
local_array_dir = local_graph_dir / "tekarx_graph_arrays"
artifact_records = graph_marker.get("artifacts", {})
copy_plan: list[tuple[Path, Path, int | None]] = []

manifest_artifact = artifact_records.get("tekarx_graph_arrays/manifest.json", {})
copy_plan.append((
    drive_array_manifest_path, local_array_dir / "manifest.json", manifest_artifact.get("size_bytes")
))
for array_name in required_arrays:
    metadata = array_manifest.get("arrays", {}).get(array_name)
    if not metadata:
        raise RuntimeError(f"Missing array metadata: {array_name}")
    filename = metadata["path"]
    artifact = artifact_records.get(f"tekarx_graph_arrays/{filename}", {})
    copy_plan.append((drive_array_dir / filename, local_array_dir / filename, artifact.get("size_bytes")))

drug_metadata_name = array_manifest["drug_metadata_path"]
drug_metadata_artifact = artifact_records.get(f"tekarx_graph_arrays/{drug_metadata_name}", {})
copy_plan.append((
    drive_array_dir / drug_metadata_name, local_array_dir / drug_metadata_name,
    drug_metadata_artifact.get("size_bytes"),
))
graph_manifest_artifact = artifact_records.get("graph_manifest.json", {})
copy_plan.append((
    drive_graph_dir / "graph_manifest.json", local_graph_dir / "graph_manifest.json",
    graph_manifest_artifact.get("size_bytes"),
))

bytes_needed = sum(source.stat().st_size for source, _, _ in copy_plan)
free_bytes = shutil.disk_usage("/content").free
print(f"Topology restore size: {bytes_needed / 2**30:.2f} GiB")
if free_bytes < bytes_needed + 2 * 2**30:
    raise RuntimeError("Less than 2 GiB of safe local headroom would remain after restore.")
for source, destination, expected_size in copy_plan:
    copy_with_progress(source, destination, expected_size=expected_size)

structures_source = DRIVE_DATA / "interim" / "drugcentral" / "structures.parquet"
if structures_source.is_file():
    copy_with_progress(
        structures_source, LOCAL_DATA / "interim" / "drugcentral" / "structures.parquet"
    )
else:
    print("WARNING: DrugCentral structures.parquet is absent; DC IDs will be used as labels.")

restore_record = {
    "stage": "full_graph_visualization_restore_complete",
    "completed_at_utc": datetime.now(UTC).isoformat(),
    "graph_checkpoint_id": GRAPH_CHECKPOINT_ID,
    "source_marker": str(marker_path),
    "copied_arrays": list(required_arrays),
    "intentionally_omitted": ["patient_x", "patient_is_death", "patient_is_hospitalization"],
}
(local_graph_dir / "visualization_restore.json").write_text(
    json.dumps(restore_record, indent=2) + "\n", encoding="utf-8"
)
print(f"Restored exact full topology checkpoint: {GRAPH_CHECKPOINT_ID}")

## 4. Build the interactive 3D view

Sampling is deterministic for the configured seed and approximately balances serious/non-serious patients. Labels come from DrugCentral when its structures table is available.

In [ ]:
visualization_dir = LOCAL_DATA / "processed" / "visualizations"
visualization_dir.mkdir(parents=True, exist_ok=True)
output_html = visualization_dir / f"{GRAPH_CHECKPOINT_ID}_{SPLIT}_3d.html"
run_live([
    sys.executable, "-m", "tekarx.cli", "visualize-graph",
    "--data-dir", str(LOCAL_DATA),
    "--graph-dir", str(local_graph_dir),
    "--layout", "3d",
    "--split", SPLIT,
    "--patients", str(PATIENTS),
    "--top-drugs", str(TOP_DRUGS),
    "--seed", str(SEED),
    "--output", str(output_html),
])

output_json = output_html.with_suffix(".json")
visualization_manifest = load_json(output_json)
record = visualization_manifest["record"]
patient_percentage = 100 * record["rendered_patients"] / record["graph_patient_nodes"]
drug_percentage = 100 * record["rendered_drugs"] / record["graph_drug_nodes"]
edge_percentage = 100 * record["rendered_edges"] / record["graph_edges"]
print(json.dumps(record, indent=2))
print("\nShare of complete graph rendered:")
print(f"  patients: {patient_percentage:.6f}%")
print(f"  drugs:    {drug_percentage:.6f}%")
print(f"  edges:    {edge_percentage:.6f}%")

In [ ]:
from google.colab import output as colab_output

if "PREVIEW_SERVER" in globals() and PREVIEW_SERVER.poll() is None:
    PREVIEW_SERVER.terminate()
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
    probe.bind(("127.0.0.1", 0))
    PREVIEW_PORT = probe.getsockname()[1]
PREVIEW_SERVER = subprocess.Popen(
    [sys.executable, "-m", "http.server", str(PREVIEW_PORT), "--directory", str(output_html.parent)],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
print(f"Serving {output_html.name} on port {PREVIEW_PORT}")
colab_output.serve_kernel_port_as_iframe(PREVIEW_PORT, path=f"/{output_html.name}", height=900)

## 5. Save the visualization in Drive

The HTML is standalone and can be opened in Chrome or Edge after downloading. The adjacent JSON file preserves graph counts, sampling parameters, and provenance.

In [ ]:
drive_visualization_dir = DRIVE_DATA / "processed" / "visualizations"
drive_visualization_dir.mkdir(parents=True, exist_ok=True)
for local_path in (output_html, output_json):
    drive_path = drive_visualization_dir / local_path.name
    copy_with_progress(local_path, drive_path)
    if sha256_file(local_path) != sha256_file(drive_path):
        raise RuntimeError(f"Drive verification failed: {drive_path}")
    print(f"VERIFIED {drive_path}")

if DOWNLOAD_TO_BROWSER:
    from google.colab import files
    files.download(str(output_html))
print("Full-graph visualization checkpoint complete.")

## If you truly need to rematerialize every feature array

Use **Stage 2: Build and persist the full graph** in `train_full_colab.ipynb`. That operation recreates `patient_x`, the XGBoost baseline, and all topology arrays and can take more than an hour. This visualization notebook deliberately restores the already verified full topology, which is faster, restart-safe, and produces the same graph structure used by the trained model.